In [ ]:
# The goal of this notebook is to prepare raw data for disease identification.
# Using the PlantVillage dataset, I implement a preprocessing pipeline designed
# to ensure model robustness and computational efficiency.

# Key Objectives:
# Data Partitioning: I split the original dataset into Training (70%), Validation (15%), and Testing (15%) sets.
# This ensures that the model is evaluated on entirely unseen data to accurately measure its generalization capabilities.

# Dynamic Resizing & Augmentation: During the Phase 1 of the project I utilize torchvision.transforms.Training
# to  apply RandomResizedCrop and RandomHorizontalFlip to artificially increase dataset variety and prevent overfitting.
# Later on this was replaced with Modern Keras way by using the "Sequential" model of layers. This is the preferred method
# in Keras as it can run on GPUs .

# Validation/Testing:
# I use CenterCrop and Resize to maintain consistency during evaluation.

# Normalization:
# Images are normalized using the ImageNet mean and standard deviation to align the data distribution
# with pre-trained architecture of ResNet.

In [ ]:
# Mounting Google Drive to Colab

from google.colab import drive
drive.mount('/content/gdrive/')

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).


In [ ]:
import os
import shutil
import random
from pathlib import Path
from PIL import Image
from collections import Counter
import pandas as pd
import tqdm

In [ ]:
# The PlantVillage dataset provides images in color, grayscale, and segmented formats.
# I used color images , as they preserve essential chromatic information
# required for accurate disease identification.


In [ ]:
# Path to original dataset
source_dir = "/content/gdrive/MyDrive/leaf_diagnosis_project/data/plantVillage/color"

# Path where split dataset will be created
target_dir = "/content/gdrive/MyDrive/leaf_diagnosis_project/data/plantVillage_split"

In [ ]:
# Checking if source images are not 256 x 256
# Statistically, checking a random sample of 1,000 images will give
# a 99% confidence level about the dataset's consistency.

source_obj_path = Path(source_dir)
IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp"}

# Collect all image paths
files = [p for p in source_obj_path.rglob("*") if p.suffix.lower() in IMG_EXTS]

print(f"Total images found: {len(files)}")

sizes = []
non_256_images = []
sample_files = random.sample(files, min(1000, len(files)))

for p in tqdm.tqdm(sample_files):
    try:
        with Image.open(p) as img:
            size = img.size  # (width, height)
            sizes.append(size)
            if size != (256, 256):
                non_256_images.append((p, size))
    except:
        continue

# Count all resolutions
size_counts = Counter(sizes)

print("\nResolution distribution:")
for size, count in size_counts.most_common():
    print(f"{size} : {count}")

print("\nNumber of images NOT 256x256:", len(non_256_images))

# If there are any non-256 images, show first 10
if non_256_images:
    print("\nExamples of non-256 images:")
    for p, size in non_256_images[:10]:
        print(p, size)

Total images found: 54305


100%|██████████| 1000/1000 [03:37<00:00,  4.59it/s]


Resolution distribution:
(256, 256) : 1000

Number of images NOT 256x256: 0


In [ ]:
#Create Train / Val / Test Folders
splits = ['train', 'val', 'test']

for split in splits:
    os.makedirs(os.path.join(target_dir, split), exist_ok=True)

In [ ]:
# Split the Dataset (Core Logic)
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

random.seed(42)  # reproducibility

for class_name in os.listdir(source_dir):
    class_path = os.path.join(source_dir, class_name)

    if not os.path.isdir(class_path):
        continue

    images = os.listdir(class_path)
    random.shuffle(images)

    total = len(images)
    train_end = int(train_ratio * total)
    val_end = train_end + int(val_ratio * total)

    splits_dict = {
        'train': images[:train_end],
        'val': images[train_end:val_end],
        'test': images[val_end:]
    }

    for split, split_images in splits_dict.items():
        split_class_dir = os.path.join(target_dir, split, class_name)
        os.makedirs(split_class_dir, exist_ok=True)

        for img in split_images:
            src = os.path.join(class_path, img)
            dst = os.path.join(split_class_dir, img)
            shutil.copy(src, dst)

In [ ]:
# Checking that generated data.
!ls /content/gdrive/MyDrive/leaf_diagnosis_project/data/plantVillage_split

test  train  val
